# 6장 — 이웃 찾기: kNN에서 벡터 DB까지 🔍

전수조사 kNN을 직접 만들고, 구역 나누기(IVF식 ANN)로 **빨라지는 것과 놓치는 것**을 재현합니다.

**이 노트북의 목표**
1. kNN 전수조사 구현 — 정확하지만 비교 횟수 = 문서 수(선형)
2. 구역 나누기 ANN 구현 — 비교 급감, 대신 경계 질문에서 진짜 1등을 놓치는 사례 재현
3. 조사 구역 수(nprobe)를 늘리면 정확도가 회복되는 것을 100개 질문으로 측정

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch6')

## 1. 실험 ① — 정직한 kNN: 몇 줄이면 끝, 비용은 선형

In [ ]:
import numpy as np
rng = np.random.default_rng(606)

def knn_full(q, docs, k=1):
    """전수조사 kNN — 모든 문서와 비교. (docs는 정규화됐다고 가정 → 점수는 내적 한 방, 2장)"""
    scores = docs @ q                     # 문서 수만큼의 내적
    return np.argsort(-scores)[:k], len(docs)

d = 64
for n in [1_000, 100_000]:
    docs = rng.standard_normal((n, d))
    docs /= np.linalg.norm(docs, axis=1, keepdims=True)
    noise = rng.standard_normal(d)
    q = docs[0] + 0.3 * noise / np.linalg.norm(noise)   # 0번 문서와 관련된 질문 (cos ≈ 0.96)
    q /= np.linalg.norm(q)
    top, comparisons = knn_full(q, docs)
    print(f'문서 {n:>7,}건 → 비교 {comparisons:>7,}번, 1등 = {top[0]}번 문서')
    assert top[0] == 0, '질문과 관련지어 만든 0번 문서가 1등이어야'

print('\n✔ 확인: 항상 정확하다 — 대신 비교 횟수가 문서 수 그대로(선형 비용)')

## 2. 실험 ② — 구역 나누기(IVF식 ANN): 웹의 함정 사례 재현

문서를 구역(클러스터)으로 미리 정리해 두고, 질문과 가까운 구역만 조사합니다.
웹 실험과 같은 배치 — 3개 구역 + 경계의 함정 점.

In [ ]:
centers = np.array([[2.5, 3.0], [7.5, 3.0], [5.0, 8.0]])
pts = np.vstack([c + 0.8 * rng.standard_normal((18, 2)) for c in centers])
pts = np.vstack([pts, [[4.2, 4.5]]])                 # 함정 점 (1구역 소속, 경계 근처)
zone = np.argmin(np.linalg.norm(pts[:, None] - centers[None], axis=2), axis=1)

def ann_search(q, nprobe=1):
    """질문과 가까운 구역 nprobe개만 조사. 반환: (찾은 문서, 비교 횟수)"""
    zorder = np.argsort(np.linalg.norm(centers - q, axis=1))[:nprobe]
    cand = np.where(np.isin(zone, zorder))[0]
    best = cand[np.argmin(np.linalg.norm(pts[cand] - q, axis=1))]
    return best, len(centers) + len(cand)

q_edge = np.array([4.7, 5.2])                        # 경계의 질문
true_best = int(np.argmin(np.linalg.norm(pts - q_edge, axis=1)))

b1, c1 = ann_search(q_edge, nprobe=1)
b2, c2 = ann_search(q_edge, nprobe=2)
print(f'진짜 1등: {true_best}번 (함정 점)')
print(f'구역 1개: {b1}번 찾음, 비교 {c1}/{len(pts)}  → {"✓ 정확" if b1 == true_best else "✗ 놓침!"}')
print(f'구역 2개: {b2}번 찾음, 비교 {c2}/{len(pts)}  → {"✓ 정확" if b2 == true_best else "✗ 놓침!"}')

assert true_best == len(pts) - 1, '경계 질문의 진짜 1등은 함정 점이어야'
assert b1 != true_best, '구역 1개만 조사하면 옆 구역의 진짜 1등을 놓친다 — 근사의 대가'
assert b2 == true_best, '구역을 2개로 늘리면 회복된다 — nprobe 손잡이'
assert c1 < len(pts), '그 대신 비교 횟수는 전수보다 적다'
print('\n✔ 확인: 빨라지는 대신 경계에서 놓친다 → 손잡이(nprobe)로 정확도·속도를 거래')

## 3. 실험 ③ — 100개 질문으로 성적 재기 (7장 예고)

질문 하나의 일화가 아니라 **비율**로 — 진짜 1등을 찾은 질문의 비율(정확도)과 평균 비교 횟수.

In [ ]:
queries = rng.uniform(0.5, 9.5, size=(100, 2))       # 지도 전역에 무작위 질문 100개

def score(nprobe):
    hits, comps = 0, 0
    for q in queries:
        tb = int(np.argmin(np.linalg.norm(pts - q, axis=1)))
        b, c = ann_search(q, nprobe)
        hits += int(b == tb)
        comps += c
    return hits / len(queries), comps / len(queries)

print(' nprobe   진짜 1등 적중률   평균 비교 (전수 = 55)')
acc = {}
for np_ in [1, 2, 3]:
    a, c = score(np_)
    acc[np_] = a
    print(f'   {np_}        {a:5.0%}            {c:.0f}')

assert acc[1] < 1.0, '구역 1개는 100%가 아니다 — 경계 질문들이 새어 나간다'
assert acc[2] >= acc[1] and acc[3] >= acc[2], '구역을 늘릴수록 적중률은 오른다 (속도를 내주고)'
assert acc[3] == 1.0, '전 구역(3개)을 보면 전수조사와 같아진다'
print('\n✔ 확인: 정확도↔속도의 거래를 숫자로 확인 — 이런 "성적표"를 체계화한 것이 7장의 recall@k')

## 4. 관찰 정리 + 🤖 AI 연결

- **kNN 전수조사**: 몇 줄이면 되고 항상 정확 — 비용이 문서 수에 정비례(선형)라는 것만 문제.
  수천~수십만 건이면 이걸로 충분한 경우가 많다 (규모가 방법을 고른다).
- **ANN**(근사 최근접 이웃): 미리 정리(구역 IVF·그래프 HNSW)해 두고 일부만 조사 —
  비교 급감, 대신 경계에서 가끔 놓침. 손잡이(nprobe·ef)로 정확도↔속도를 거래.
- **벡터 DB** = 임베딩 저장 + ANN 색인 + 메타데이터 필터(권한·부서·날짜) —
  FAISS·pgvector·Chroma·Pinecone 전부 같은 정체. 점수 계산은 여전히 2장의 내적.

**다음 챕터**: 7장 — 청킹과 검색 품질. 문서를 얼마나 잘게 자를까, 몇 개를 꺼낼까(top-k),
그리고 잘 찾는지는 어떻게 채점할까 — **recall@k**.